# Gesamtdokument: Face Model Lab

Stand: 2026-06-15

Ziel des Projekts ist ein praktisch nutzbares, selbst trainiertes Modell zur Gesichtserkennung und anschliessenden Video-Anonymisierung. Verglichen wurden YOLOv8m, Faster R-CNN, RetinaNet und FCOS auf WIDER FACE. COCO-pretrained Modelle wurden nur als Baseline aufgenommen, weil sie nicht face-spezifisch trainiert sind.

## Wichtigste Erkenntnisse

- Das beste reine Recall-Ergebnis im aktuellen 300-Bild-Test hat FCOS mit `0.542`, ist aber mit `129.7 ms/Bild` fuer Video sehr langsam.
- YOLOv8m bs2 ep15 ist aktuell der beste praktische Kompromiss: `0.536` Recall bei `40.9 ms/Bild`.
- RetinaNet bs2 ep10 ist ueberraschend stark: `0.524` Recall bei `35.4 ms/Bild`.
- Faster R-CNN bs2 ep10 liegt fast gleichauf mit RetinaNet: `0.523` Recall bei `52.0 ms/Bild`.
- COCO-pretrained Baselines erkennen Gesichter kaum sinnvoll, weil COCO keine Face-Box-Aufgabe ist.
- Fuer Video-Anonymisierung ist Recall wichtiger als reine Geschwindigkeit: ein schnelles Modell, das Gesichter verpasst, ist fuer Datenschutz riskant.
- Die bisherigen RetinaNet/FCOS-Laeufe mit `reduction=50` sind gute Laborvergleiche, aber noch keine finalen Produktionsmodelle.

## Aktueller Modellvergleich

| Modell | Trainingsdaten | Recall | ms/Bild | Einschaetzung |
|---|---:|---:|---:|---|
| FCOS bs2 ep10 | reduction=50, 258 Bilder | 0.542 | 129.7 | bester Recall, aber nicht echtzeitnah |
| YOLOv8m bs2 ep15 | groesserer YOLO-Lauf | 0.536 | 40.9 | bester Praxis-Kompromiss |
| RetinaNet bs2 ep10 | reduction=50, 258 Bilder | 0.524 | 35.4 | starke Torchvision-Baseline |
| Faster R-CNN bs2 ep10 | reduction=2, 6440 Bilder | 0.523 | 52.0 | robuste Two-Stage-Baseline |
| YOLOv8m bs2 ep10 | groesserer YOLO-Lauf | 0.260 | 16.6 | schnell, aber schwacher Checkpoint |

Dateien:

- Evaluation CSV: [`model_results/evaluation_20260615_095026.csv`](../model_results/evaluation_20260615_095026.csv)
- Evaluation JSON: [`model_results/evaluation_20260615_095026.json`](../model_results/evaluation_20260615_095026.json)
- YOLO best: [`trained_models/yolo_yolov8m_widerface_rocm_bs2_ep15.pt`](../trained_models/yolo_yolov8m_widerface_rocm_bs2_ep15.pt)
- RetinaNet: [`trained_models/retinanet_resnet50_fpn_rocm_bs2_ep10.pth`](../trained_models/retinanet_resnet50_fpn_rocm_bs2_ep10.pth)
- FCOS: [`trained_models/fcos_resnet50_fpn_rocm_bs2_ep10.pth`](../trained_models/fcos_resnet50_fpn_rocm_bs2_ep10.pth)
- Faster R-CNN: [`trained_models/fasterrcnn_resnet50_fpn_rocm_bs2_ep10.pth`](../trained_models/fasterrcnn_resnet50_fpn_rocm_bs2_ep10.pth)

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path('..').resolve()
eval_csv = ROOT / 'model_results' / 'evaluation_20260615_095026.csv'
df = pd.read_csv(eval_csv)
df.sort_values(['recall', 'ms_per_image'], ascending=[False, True])

## Grafiken

### Validierungs-Recall

![Validation Recall](../model_results/plots/validation_recall_20260615_095026.png)

### Latenz

![Validation Latency](../model_results/plots/validation_latency_20260615_095026.png)

### Trainingsdaten vs Validierungsdaten

![Train vs Val](../model_results/plots/dataset_train_vs_val_20260615_095026.png)

## Trainierte Videos und Blur-Vergleich

Erstellt wurden 30s-Videos aus `Videos/Testmaterial Gesichter.mp4`. Weil der Quellclip interlaced war, wurde zunaechst eine progressive 30s-Version erstellt.

- Original progressiv: [`Videos/lab_outputs/original_30s_testmaterial_gesichter_progressive.mp4`](../Videos/lab_outputs/original_30s_testmaterial_gesichter_progressive.mp4)
- Anonymisiert, ovale weiche Gaussian-Maske: [`Videos/lab_outputs/anonymisiert_30s_oval_gaussian.mp4`](../Videos/lab_outputs/anonymisiert_30s_oval_gaussian.mp4)
- Anonymisiert, Pixelate: [`Videos/lab_outputs/anonymisiert_30s_pixelate.mp4`](../Videos/lab_outputs/anonymisiert_30s_pixelate.mp4)
- Vergleich Original vs Gaussian: [`Videos/lab_outputs/vergleich_30s_original_vs_oval_gaussian.mp4`](../Videos/lab_outputs/vergleich_30s_original_vs_oval_gaussian.mp4)
- Vergleich Original vs Pixelate: [`Videos/lab_outputs/vergleich_30s_original_vs_pixelate.mp4`](../Videos/lab_outputs/vergleich_30s_original_vs_pixelate.mp4)

Gaussian/Oval wirkt optisch sauberer und weniger hart, ist aber wesentlich rechenintensiver. Pixelate ist schneller und sichtbar eindeutiger anonymisiert, wirkt aber grober.

## Reduction 2 vs Reduction 50

`reduction=50` bedeutet: Es wird nur jedes 50. Trainingsbild verwendet. Bei 12.880 WIDER-FACE-Trainingsbildern sind das 258 Bilder.

`reduction=2` bedeutet: Es wird jedes zweite Bild verwendet. Das sind 6.440 Bilder, also etwa 25x so viele Bilder wie bei `reduction=50`.

Ein Lauf mit `reduction=2` waere deutlich aussagekraeftiger, weil:

- mehr Posen, Lichtverhaeltnisse, Groessen und Verdeckungen vorkommen,
- kleine Gesichter und Crowd-Szenen besser gelernt werden,
- Overfitting auf wenige Bilder weniger wahrscheinlich ist,
- die Validierungsleistung stabiler interpretierbar wird.

Er waere aber nicht automatisch 25x besser. Er ist eher 25x groesser und wahrscheinlich deutlich robuster. Die aktuellen `reduction=50`-Runs zeigen, welches Modell Potenzial hat. Ein finaler Modellvergleich sollte mindestens mit `reduction=10` oder `reduction=5` wiederholt werden; `reduction=2` ist fuer finale Torchvision-Baselines sinnvoll, wenn die Laufzeit akzeptabel ist.

## Warum gleiche Datensaetze wichtig sind

Modelle duerfen nur fair verglichen werden, wenn Trainings- und Validierungsbedingungen gleich sind:

- gleiche Trainingsbilder oder zumindest gleiche Datenmenge,
- gleiche Validierungsbilder,
- gleiche IoU-/Confidence-Schwellen,
- gleiche Bildgroesse oder bewusst dokumentierte Bildgroesse,
- gleiche Zieldefinition: Face-Box, nicht Person-Box.

Sonst vergleicht man nicht nur Architekturen, sondern gleichzeitig Datenauswahl, Trainingsdauer, Augmentierung, Schwellenwerte und Laufzeitumgebung. Das fuehrt leicht zu falschen Schlussfolgerungen.

Die aktuellen Ergebnisse zeigen: FCOS/RetinaNet wurden mit `reduction=50` trainiert, Faster R-CNN mit `reduction=2`, YOLO mit eigenem YOLO-Datensatzlauf. Deshalb ist das Ranking ein sehr guter Zwischenstand, aber noch kein endgueltiger wissenschaftlicher Benchmark.

## Gewichteter Vergleich

Fuer den praktischen Use Case Gesichtsanonymisierung reicht ein einzelner Wert nicht. Sinnvoll ist ein gewichteter Score.

Beispiel fuer Datenschutz/Video:

- Recall: 60 Prozent, weil verpasste Gesichter das groesste Risiko sind.
- Latenz: 25 Prozent, weil Video praktisch verarbeitbar bleiben muss.
- Pipeline-Komfort: 10 Prozent, weil Tracking, Export und Deployment Aufwand verursachen.
- Trainingskosten: 5 Prozent, weil lange Laeufe Entwicklung verlangsamen.

Eine einfache Formel waere:

`score = 0.60 * normalized_recall + 0.25 * normalized_speed + 0.10 * pipeline_score + 0.05 * training_score`

Dabei sollte `normalized_speed` nicht linear in ms/Bild genutzt werden, sondern als relative Geschwindigkeit innerhalb des Vergleichs. Beispiel: das schnellste Modell bekommt 1.0, das langsamste 0.0. Fuer Datenschutz sollte Recall aber immer dominieren.

In [ ]:
# Beispiel: einfache gewichtete Auswertung der aktuellen CSV.
# Pipeline- und Trainingsscores sind hier bewusst grobe Projektscores.
import pandas as pd

df = pd.read_csv(eval_csv).copy()
df = df[~df['model'].str.startswith('BASELINE_PRETRAINED_COCO')].copy()
df['recall_norm'] = df['recall'] / df['recall'].max()
df['speed_norm'] = (df['ms_per_image'].max() - df['ms_per_image']) / (df['ms_per_image'].max() - df['ms_per_image'].min())

pipeline_scores = {
    'yolo_yolov8m_widerface_rocm_bs2_ep15.pt': 1.0,
    'yolov8m_widerface_rocm_bs2_ep10.pt': 1.0,
    'retinanet_resnet50_fpn_rocm_bs2_ep10.pth': 0.65,
    'fasterrcnn_resnet50_fpn_rocm_bs2_ep10.pth': 0.55,
    'fcos_resnet50_fpn_rocm_bs2_ep10.pth': 0.60,
}
training_scores = {
    'yolo_yolov8m_widerface_rocm_bs2_ep15.pt': 0.80,
    'yolov8m_widerface_rocm_bs2_ep10.pt': 0.80,
    'retinanet_resnet50_fpn_rocm_bs2_ep10.pth': 0.60,
    'fasterrcnn_resnet50_fpn_rocm_bs2_ep10.pth': 0.45,
    'fcos_resnet50_fpn_rocm_bs2_ep10.pth': 0.35,
}

df['pipeline_score'] = df['model'].map(pipeline_scores).fillna(0.5)
df['training_score'] = df['model'].map(training_scores).fillna(0.5)
df['weighted_score'] = (
    0.60 * df['recall_norm'] +
    0.25 * df['speed_norm'] +
    0.10 * df['pipeline_score'] +
    0.05 * df['training_score']
)
df[['model', 'recall', 'ms_per_image', 'weighted_score']].sort_values('weighted_score', ascending=False)

## Staerken von Torchvision

Torchvision ist in diesem Projekt besonders stark als kontrollierte Forschungs- und Baseline-Umgebung:

- Standardmodelle wie Faster R-CNN, RetinaNet und FCOS sind direkt verfuegbar.
- COCO-pretrained Backbones sind leicht nutzbar.
- Die Modellkoepfe koennen sauber auf `face` als eigene Klasse umgebaut werden.
- Loss-Komponenten und Trainingsloop sind transparent.
- Checkpoints, LR-Scheduler, Gradient-Clipping und eigene Datasets sind gut kontrollierbar.

Der Nachteil: Fuer eine Video-Pipeline ist Torchvision weniger komfortabel als YOLO/Ultralytics. Tracking, Export, Postprocessing und Echtzeit-Deployment sind bei YOLO meist einfacher. Torchvision ist also hervorragend fuer Modellvergleich und Forschung, aber nicht automatisch die beste Produktionspipeline.

## Wie koennte man die Trainingszeit von 20h optimieren?

Die GPU ist stark, aber Training ist nicht nur GPU-Rechenleistung. Bei WIDER FACE bremsen oft variable Bildgroessen, viele kleine Boxen, CPU-Dataloader, Python/Postprocessing und unguenstige Batch-Zusammenstellung.

Sinnvolle Optimierungen:

- Bilder vorab auf feste Maximalgroesse preprocessen, damit nicht jedes Training dynamisch skaliert.
- Aspect-ratio-bucketing nutzen, damit aehnliche Bildformen in einem Batch landen.
- DataLoader optimieren: `num_workers` testen, `persistent_workers=True`, `pin_memory` pruefen.
- Mixed Precision auf ROCm testen, wenn stabil: `torch.autocast(device_type='cuda', dtype=torch.float16)` kann helfen, muss aber validiert werden.
- Backbone zuerst einfrieren und nur den Detection Head trainieren, danach Backbone teilweise auftauen.
- Mit `reduction=10` oder `reduction=5` Modellwahl treffen, erst den Gewinner mit `reduction=2` oder voll trainieren.
- Early Stopping ueber Validierungs-Recall nutzen, nicht stur Epochen fertig trainieren.
- Checkpoint-Auswertung pro Epoche automatisieren, damit man den besten Epoch-Checkpoint nimmt.
- Sehr grosse Crowd-Bilder separat behandeln oder cappen, weil sie Training stark verlangsamen koennen.
- Fuer Video final YOLO oder ein spezialisiertes Face-Detector-Modell exportieren und Torchvision eher als Qualitaetsbaseline nutzen.

Wichtig: Eine groessere Batchsize ist nicht automatisch schneller. Wenn einzelne Bilder sehr viele Boxen haben oder die CPU/Postprocessing bremst, steigt die Epochendauer trotz mehr VRAM.

## Sind die Modelle echtzeitfaehig?

Aus `ms/Bild` laesst sich grob FPS abschaetzen:

- YOLOv8m ep15: 40.9 ms/Bild -> ca. 24 FPS fuer reine Detection.
- RetinaNet: 35.4 ms/Bild -> ca. 28 FPS fuer reine Detection.
- Faster R-CNN: 52.0 ms/Bild -> ca. 19 FPS.
- FCOS: 129.7 ms/Bild -> ca. 7.7 FPS.

Das ist aber nur Detection auf Validierungsbildern. Eine echte Video-Pipeline hat zusaetzlich Decoding, Resize, Tracking, Blur, Encoding und ggf. Audio-Muxing. Fuer 1080p/50fps Echtzeit ist aktuell keines der Modelle sicher ausreichend, wenn jedes Frame voll verarbeitet wird.

Praktisch relevant sind sie trotzdem:

- Offline-Anonymisierung ist mit YOLOv8m und RetinaNet realistisch.
- Near-realtime ist moeglich, wenn nur jedes zweite oder dritte Frame detektiert wird und dazwischen Tracking/Interpolation genutzt wird.
- FCOS ist aktuell eher Forschungs-/Qualitaetsbaseline, nicht erste Wahl fuer Video.
- Fuer produktive Echtzeit waeren YOLO mit Tracking, kleineres Modell, niedrigere Aufloesung oder ein spezialisierter Face-Detector sinnvoller.

## Optimierungsplan fuer bessere Modelle

Empfohlene Reihenfolge:

1. Validierung fixieren: gleiche 500-1000 Validierungsbilder, gleiche Schwellenwerte, gleiche Auswertung.
2. Schnellen Architekturvergleich mit `reduction=10` fuer YOLO, RetinaNet, FCOS, Faster R-CNN.
3. Beste 2 Modelle mit `reduction=2` trainieren.
4. Pro Epoche evaluieren und besten Checkpoint nach Validierungs-Recall und Latenz waehlen.
5. Fuer YOLO `imgsz`, `conf`, Augmentierung, kleine Gesichter und Schwellenwerte optimieren.
6. Fuer Torchvision LR, Backbone-Freezing, Scheduler, Bildgroesse und Anchor/FCOS-Parameter testen.
7. Fuer Video nicht nur Recall messen, sondern auch verpasste Gesichter pro Minute, False Positives und Blur-Stabilitaet.

Fuer Datenschutz sollte das Ziel nicht nur ein hoher Durchschnitts-Recall sein, sondern besonders wenige komplett verpasste Gesichter in schwierigen Szenen.